# សេម៉ាន្ទិក កឺណែល 

ក្នុងគំរូកូដនេះ អ្នកនឹងប្រើ [សេម៉ាន្ទិក កឺណែល](https://aka.ms/ai-agents-beginners/semantic-kernel) ស៊ុមគ្រឹះ AI ដើម្បីបង្កើតភ្នាក់ងារងាយៗ។

គោលបំណងនៃគំរូនេះគឺដើម្បីបង្ហាញជំហាននានាដែលនឹងត្រូវបានអនុវត្តនៅក្នុងគំរូកូដបន្ថែមផ្សេងទៀតពេលអនុវត្តលំនាំភ្នាក់ងារផ្សេងៗ។


## នាំចូលកញ្ចប់ Python ដែលចាំបាច់


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## ការបង្កើត Client

នៅក្នុងឧទាហរណ៍នេះ យើងនឹងប្រើ [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) ដើម្បីចូលប្រើ LLM។

`ai_model_id` ត្រូវបានកំណត់ជា `gpt-4o-mini`។ សាកល្បងប្ដូរម៉ូដែលទៅមួយផ្សេងទៀតដែលមាននៅក្នុងទីផ្សារ GitHub Models ដើម្បីសង្កេតឃើញលទ្ធផលខុសគ្នា។

ដើម្បីប្រើ `Azure Inference SDK` ដែលត្រូវការសម្រាប់ `base_url` សម្រាប់ GitHub Models យើងនឹងប្រើ `OpenAIChatCompletion` connector ក្នុង Semantic Kernel។ ក៏មាន [ឧបករណ៍ភ្ជាប់ដែលមាន](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) ផង ដែលអនុញ្ញាតឲ្យអ្នកប្រើ Semantic Kernel ជាមួយអ្នកផ្គត់ផ្គង់ម៉ូដែលផ្សេងៗ។


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## ការបង្កើតភ្នាក់ងារ

នៅទីនេះ យើងបង្កើតភ្នាក់ងារដែលមានឈ្មោះ `TravelAgent`។

នៅក្នុងឧទាហរណ៍នេះ យើងប្រើសេចក្តីណែនាំមូលដ្ឋាន។ អ្នកអាចកែសម្រួលសេចក្តីណែនាំទាំងនេះ ដើម្បីសង្កេតមើលថាប្រព្រឹត្តិការណ៍របស់ភ្នាក់ងារប្រែប្រួលយ៉ាងដូចម្តេច។


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## ការរត់ភ្នាក់ងារ

ឥឡូវនេះយើងអាចអនុវត្តភ្នាក់ងារ ដោយរៀបចំ `ChatHistory` និងបញ្ចូល `system_message` នៅក្នុងវា។ យើងនឹងប្រើ `AGENT_INSTRUCTIONS` ដែលយើងបានកំណត់មុននេះ។

Once these are set up, we create `user_inputs`, which represent what the user is sending to the agent. In this example, the message is set to `Plan me a sunny vacation`.

អ្នកអាចកែសម្រួលសារនេះ ដើម្បីមើលថាតើភ្នាក់ងារឆ្លើយតបបានខុសគ្នាយ៉ាងដូចម្តេច។


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំរក្សាឱ្យមានភាពត្រឹមត្រូវ សូមចំណាំថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬមានភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាម្ចាស់គួរត្រូវបានគេចាត់ទុកជាប្រភពដែលអាចទុកចិត្តបាន។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយមនុស្សដែលមានវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
